In [1]:
import numpy as np
import torch as torch
import os
import copy
import random
from scipy.optimize import nnls,curve_fit,leastsq
from sklearn.preprocessing import PolynomialFeatures
import time

In [2]:
def parameter_limit(x,up,down):
    mask=(x>up)
    x[mask]=up
    mask=(x<down)
    x[mask]=down
    return x

In [3]:
def FIT(val_signals):
    s0=val_signals[:,0].unsqueeze(1)
    s50=val_signals[:,1].unsqueeze(1)
    s150=val_signals[:,2].unsqueeze(1)
    s500=val_signals[:,3].unsqueeze(1)

    Dslow=torch.log(s150/s500)/(500-150)
    Dslow=parameter_limit(Dslow,up=1e8,down=0)
    fslow=(s150/s0)*torch.exp(150*Dslow)
    fslow=parameter_limit(fslow,up=1,down=0)
    ffast=1-fslow
    temp=((s50/s0)-fslow*torch.exp(-30*Dslow))/(ffast+1e-8)
    temp=parameter_limit(temp,up=1e8,down=1e-8)
    Dfast=(-1/30)*torch.log(temp)
    Dfast=parameter_limit(Dfast,up=1e8,down=0)
    generate_parameters=torch.cat((Dslow,Dfast,fslow,ffast),dim=1)
    return generate_parameters


In [4]:
data_dict=torch.load(r"G:\zhouxinxiang\MRL_HN\simulation\data_dict.pth")
parameter_dict=torch.load(r"G:\zhouxinxiang\MRL_HN\simulation\parameter_dict.pth")

error_dict=torch.load(r"G:\zhouxinxiang\MRL_HN\simulation\error_dict.pth")

val_signals,val_parameters=data_dict['val']['Average']
      


since = time.time()

generate_parameters=FIT(val_signals)

time_elapsed = time.time() - since

parameter_dict['segmentedanalytical'+'val']=generate_parameters
error_dict['segmentedanalytical'+'val']=generate_parameters-np.array(val_parameters)


In [10]:
torch.save(parameter_dict,r"G:\zhouxinxiang\MRL_HN\simulation\parameter_dict.pth")
torch.save(error_dict,r"G:\zhouxinxiang\MRL_HN\simulation\error_dict.pth")